In [4]:
!pip install -q \
google-genai \
faiss-cpu \
pandas \
numpy \
pypdf \
pdfplumber \
openpyxl \
python-dotenv

In [5]:
import faiss
import pandas as pd
import numpy as np
import pdfplumber

from google import genai

print("Everything installed successfully!")

Everything installed successfully!


In [6]:
from google.colab import files

uploaded = files.upload()

Saving financial_statements.pdf to financial_statements.pdf
Saving annual_report.pdf to annual_report.pdf


In [7]:
import pdfplumber

def extract_pdf_text(pdf_path):
    text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text

In [8]:
annual_text = extract_pdf_text("annual_report.pdf")

financial_text = extract_pdf_text("financial_statements.pdf")

print("Annual Report Characters:", len(annual_text))

print("Financial Statements Characters:", len(financial_text))

Annual Report Characters: 273013
Financial Statements Characters: 7864


In [9]:
import pdfplumber
import pandas as pd

tables = []

with pdfplumber.open("financial_statements.pdf") as pdf:

    for page in pdf.pages:

        extracted_tables = page.extract_tables()

        for table in extracted_tables:

            if table:
                df = pd.DataFrame(table)
                tables.append(df)

print("Tables Found:", len(tables))

Tables Found: 4


In [10]:
with pd.ExcelWriter("financial_data.xlsx") as writer:

    for i, table in enumerate(tables):

        table.to_excel(
            writer,
            sheet_name=f"Table_{i+1}",
            index=False
        )

print("Excel Created Successfully!")

Excel Created Successfully!


In [11]:
excel = pd.ExcelFile("financial_data.xlsx")

print(excel.sheet_names)

['Table_1', 'Table_2', 'Table_3', 'Table_4']


In [12]:
import pandas as pd

excel = pd.ExcelFile("financial_data.xlsx")

all_excel_text = ""

for sheet in excel.sheet_names:

    df = pd.read_excel(
        "financial_data.xlsx",
        sheet_name=sheet
    )

    all_excel_text += df.to_string(index=False)

print(all_excel_text[:1500])

                                           0          1   2          3   4          5   6          7
                                    Products   $ 73,716 NaN   $ 69,958 NaN  $ 307,003 NaN  $ 294,866
                                    Services     28,750 NaN     24,972 NaN    109,158 NaN    96 ,169
                         Total net sales (1)    102,466 NaN     94,930 NaN    416,161 NaN    391,035
                              Cost of sales:        NaN NaN        NaN NaN        NaN NaN        NaN
                                         NaN        NaN NaN        NaN NaN        NaN NaN        NaN
                                    Products     47,019 NaN     44,566 NaN    194,116 NaN    185,233
                                    Services      7,106 NaN      6,485 NaN     26,844 NaN     25,119
                         Total cost of sales     54,125 NaN     51,051 NaN    220,960 NaN    210,352
                                Gross margin     48,341 NaN     43,879 NaN    195,201 NaN  

In [13]:
all_text = annual_text + "\n\n" + all_excel_text

print(len(all_text))

287483


In [14]:
def create_chunks(text, chunk_size=1000, overlap=200):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [15]:
chunks = create_chunks(all_text)

print("Total Chunks:", len(chunks))

print(chunks[0])

Total Chunks: 360
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from to .
Commission File Number: 001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California 94-2404110
(State or other jurisdiction (I.R.S. Employer Identification No.)
of incorporation or organization)
One Apple Park Way
Cupertino, California 95014
(Address of principal executive offices) (Zip Code)
(408) 996-1010
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Trading
Title of each class symbol(s) Name of each exchange on which registered
Common Stock, $0.00001 par value per share AAPL The Nasdaq Stock Market LLC
0.000% Notes due 2025 

In [ ]:
from google import genai

client = genai.Client(
    api_key="YOUR_GEMINI_API_KEY"
)

print("Gemini Connected!")

Gemini Connected!


In [ ]:
from google import genai

client = genai.Client(
    api_key="YOUR_GEMINI_API_KEY"
)

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [18]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

(360, 384)


In [19]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors:", index.ntotal)

Vectors: 360


In [20]:
def search(query, top_k=5):
    query_embedding = embedding_model.encode([query]).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [21]:
results = search("What was Apple's revenue?")

for i, chunk in enumerate(results):
    print("=" * 80)
    print(f"Result {i+1}")
    print(chunk[:1000])

Result 1
e Inc. | 2025 Form 10-K | 28
Apple Inc.
CONSOLIDATED STATEMENTS OF OPERATIONS
(In millions, except number of shares, which are reflected in thousands, and per-share amounts)
Years ended
September 27, September 28, September 30,
2025 2024 2023
Net sales:
Products $ 307,003 $ 294,866 $ 298,085
Services 109,158 96,169 85,200
Total net sales 416,161 391,035 383,285
Cost of sales:
Products 194,116 185,233 189,282
Services 26,844 25,119 24,855
Total cost of sales 220,960 210,352 214,137
Gross margin 195,201 180,683 169,148
Operating expenses:
Research and development 34,550 31,370 29,915
Selling, general and administrative 27,601 26,097 24,932
Total operating expenses 62,151 57,467 54,847
Operating income 133,050 123,216 114,301
Other income/(expense), net (321) 269 (565)
Income before provision for income taxes 132,729 123,485 113,736
Provision for income taxes 20,719 29,749 16,741
Net income $ 112,010 $ 93,736 $ 96,995
Earnings per share:
Basic $ 7.49 $ 6.11 $ 6.16
Diluted $ 7.46 

In [22]:
def ask_financial_agent(question):

    docs = search(question)

    context = "\n\n".join(docs)

    prompt = f"""
You are an AI Financial Assistant.

You must answer ONLY from the provided context.

If the context contains numerical values, include them clearly.

If multiple years are available, present them in a table.

If the answer truly doesn't exist in the context, reply:
"I couldn't find that information in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="models/gemini-3.5-flash",
        contents=prompt
    )

    return response.text

In [23]:
print(ask_financial_agent("What was Apple's total revenue?"))

Based on the provided context, Apple's total net sales (total revenue) for the fiscal years 2025, 2024, and 2023 were as follows:

| Fiscal Year Ended | Total Net Sales (in millions) |
| :--- | :--- |
| September 27, 2025 | $416,161 |
| September 28, 2024 | $391,035 |
| September 30, 2023 | $383,285 |


In [24]:
print(ask_financial_agent("What was Apple's net income?"))

Based on the provided context, Apple's net income for the fiscal years 2025, 2024, and 2023 was as follows (in millions):

| Fiscal Year Ended | Net Income |
| :--- | :--- |
| **September 27, 2025** | $112,010 million |
| **September 28, 2024** | $93,736 million |
| **September 30, 2023** | $96,995 million |


In [25]:
permissions = {
    "CEO": [],
    "CTO": [
        "salary",
        "compensation",
        "headcount"
    ],
    "Intern": [
        "salary",
        "compensation",
        "headcount",
        "executive",
        "cash",
        "investment",
        "acquisition"
    ]
}

In [26]:
def has_permission(role, question):

    question = question.lower()

    restricted = permissions[role]["restricted_keywords"]

    for keyword in restricted:

        if keyword.lower() in question:
            return False

    return True

In [27]:
def financial_chat(role, question):

    previous = get_previous_feedback(question)

    if previous:
        print("📌 Previous Feedback Found:")
        print("Rating:", previous["rating"])
        print("-" * 50)

    restricted = permissions.get(role, [])

    question_lower = question.lower()

    for keyword in restricted:
        if keyword in question_lower:
            return f"""
❌ ACCESS DENIED

Role : {role}

You are not authorized to access information related to '{keyword}'.
"""

    docs = search(question)


    filtered_docs = []

    for chunk in docs:

        if not any(keyword in chunk.lower() for keyword in restricted):
            filtered_docs.append(chunk)


    if len(filtered_docs) == 0:
        return "No accessible information found for your role."

    context = "\n\n".join(filtered_docs)

    prompt = f"""
You are an AI Financial Assistant.

Use ONLY the information provided in the context.

Instructions:
- Answer only from the context.
- If numbers are present, include them.
- If multiple years are available, show them in a table.
- Do not make assumptions.
- If the answer is not present, reply:
'I couldn't find this information in the provided financial documents.'

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="models/gemini-3.5-flash",
        contents=prompt
    )

    return response.text

In [31]:
import json
import os

FEEDBACK_FILE = "feedback.json"

if not os.path.exists(FEEDBACK_FILE):
    with open(FEEDBACK_FILE, "w") as f:
        json.dump([], f)

print("Feedback file created.")

Feedback file created.


In [32]:
def save_feedback(question, answer, rating):

    with open(FEEDBACK_FILE, "r") as f:
        feedback = json.load(f)

    feedback.append({
        "question": question,
        "answer": answer,
        "rating": rating
    })

    with open(FEEDBACK_FILE, "w") as f:
        json.dump(feedback, f, indent=4)

    print("Feedback saved!")

In [34]:
with open("feedback.json") as f:
    print(f.read())

[]


In [35]:
def get_previous_feedback(question):

    with open(FEEDBACK_FILE) as f:
        feedback = json.load(f)

    for item in feedback:
        if item["question"].lower() == question.lower():
            return item

    return None

In [36]:
import faiss

faiss.write_index(index, "financial_index.faiss")

print("FAISS Index Saved")

FAISS Index Saved


In [37]:
import json

with open("chunks.json", "w") as f:
    json.dump(chunks, f)

print("Chunks Saved")

Chunks Saved


In [38]:
from google.colab import files

files.download("financial_index.faiss")
files.download("chunks.json")
files.download("feedback.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
!pip install -q gradio
import gradio as gr
import traceback

last_question = ""
last_answer = ""

def chat(role, question):
    global last_question, last_answer

    try:
        last_question = question

        answer = financial_chat(role, question)

        last_answer = answer

        return answer

    except Exception as e:

        if "429" in str(e):
            return """
# ⚠️ Gemini API Limit Reached

The free Gemini API quota has been exhausted.

Please wait about one minute and try again.
"""

        traceback.print_exc()

        return f"❌ {e}"


def good_feedback():

    if last_question != "":
        save_feedback(
            last_question,
            last_answer,
            "👍"
        )

    return "✅ Thank you! Positive feedback recorded."


def bad_feedback():

    if last_question != "":
        save_feedback(
            last_question,
            last_answer,
            "👎"
        )

    return "✅ Feedback recorded."


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI Financial Assistant"
) as demo:

    gr.Markdown("""
# 💰 AI Financial Assistant

### Retrieval-Augmented Generation (RAG)

Powered by

- 📄 Apple Annual Report
- 📊 Financial Statements
- 🤖 Gemini 3.5 Flash
- 🔍 FAISS Vector Search
- 🔐 Role Based Access Control
""")

    with gr.Row():

        with gr.Column(scale=1):

            role = gr.Dropdown(
                ["CEO","CTO","Intern"],
                value="CEO",
                label="👤 Select Role"
            )

            gr.Markdown("### Example Questions")

            gr.Markdown("""
- What was Apple's total revenue?

- What was Apple's net income?

- How much cash did Apple have?

- What were Apple's operating expenses?

- What was the gross margin?
""")

        with gr.Column(scale=3):

            question = gr.Textbox(
                label="💬 Ask a Question",
                placeholder="Ask about Apple's financial report..."
            )

            ask = gr.Button(
                "🚀 Ask AI",
                variant="primary"
            )

            answer = gr.Markdown()

            with gr.Row():

                good = gr.Button("👍 Helpful")

                bad = gr.Button("👎 Not Helpful")

            status = gr.Markdown()

    ask.click(
        chat,
        inputs=[role, question],
        outputs=answer
    )

    good.click(
        good_feedback,
        outputs=status
    )

    bad.click(
        bad_feedback,
        outputs=status
    )

demo.launch(debug=True)

/tmp/ipykernel_10897/1888183310.py:60: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://395908463d6ec44457.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/tmp/ipykernel_10897/1888183310.py", line 14, in chat
    answer = financial_chat(role, question)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_10897/166651164.py", line 62, in financial_chat
    response = client.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 6549, in generate_content
    response = self._generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 4977, in _generate_content
    response = self._api_client.request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1650, in request
    response = self._request(http_request, http_options, stream=False)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-

Feedback saved!
Feedback saved!
Feedback saved!
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://395908463d6ec44457.gradio.live
